# Ming-3.0-tts 0.5B — multilingual TTS on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/Ming-3.0-tts-0.5B.ipynb)

[`inclusionAI/Ming-3.0-tts-0.5B`](https://huggingface.co/inclusionAI/Ming-3.0-tts-0.5B) is a dense multilingual text-to-speech model (MIT, released ~22 Sep 2026). The language backbone is Qwen2.5-0.5B; a DiT / flow-matching head, audio tokenizer, speaker encoder, and text tokenizer ship in the same checkpoint. The Hub file is about 3.0 GB and lists ~1.48B parameters in total. Output is 44.1 kHz. Languages: Chinese, English, German, Spanish, French, Korean, Japanese, plus Sichuanese and Cantonese dialects.

The Hugging Face repo is **weights only**. Inference is the GitHub package [`inclusionAI/Ming-3.0-tts`](https://github.com/inclusionAI/Ming-3.0-tts): Python API `ming_tts.MingTTS` and CLI `infer.py`. This notebook clones that repo, pins a tested commit, and calls `MingTTS.synthesize` the way the [task cookbook](https://github.com/inclusionAI/Ming-3.0-tts/blob/main/cookbooks/instruction_en.md) does.

**Licence:** MIT. Do not clone an identifiable person's voice without consent. Label the audio as synthetic if it could be mistaken for a real recording.

**Runtime:** *Runtime > Change runtime type > T4 GPU*. The free T4 has 16 GB of VRAM and no native bfloat16. The upstream loader and `generate()` hardcode `torch.bfloat16` (they were measured on H800 / H20). A blanket rewrite of those casts to **float16** overflows in the DiT / flow-matching sampler (CFG at 2.0) and the audio VAE. Keeping only the audio stack in float32 is not enough: Qwen last-layer hidden states in float16 hit the 65504 range, become Inf, and poison CFG (`pred + (pred - null) * 2`) so `speak()` still returns a non-finite waveform after hundreds of steps. This notebook loads weights as float16 (Colab system RAM), then runs the **entire generate path in float32** — LLM, DiT / flow, VAE, aggregator, speaker / stop heads. Autocast is off below compute capability 8.0. Hidden states are cast to float32 before the flow head; VAE decode is wrapped in `autocast(enabled=False)` and forced to float32 so a latent cannot silently pull the decoder into float16. On an L4 or A100 the upstream bfloat16 path is left alone. Attention stays **SDPA** (`flash-attn` is optional in the README and is not installed). After the float32 upcast expect roughly 6 GB allocated at load and 8–12 GB peak for a short clip.

Tasks cannot be mixed. Each `synthesize` call is one of `TTS`, `IP`, `VOICE_DESIGN`, `EMOTION`, `BASIC`, or `MULTILINGUAL`. Inline tags such as `[Laughter]` and `[Sigh]` live in the text and work on the `TTS` task.

**Not in this notebook:** `pynini` text normalization (no reliable pip wheel; construct `MingTTS(..., text_normalization=False)`), FlashAttention 2, the MoE sibling `Ming-3.0-tts-tiny`, dialect / podcast tasks, and cloning a real person's voice.

**Links:** [model card](https://huggingface.co/inclusionAI/Ming-3.0-tts-0.5B) · [GitHub](https://github.com/inclusionAI/Ming-3.0-tts) · [cookbook](https://github.com/inclusionAI/Ming-3.0-tts/blob/main/cookbooks/instruction_en.md) · [blog & demos](https://nomorecoding.github.io/Ming-3.0-tts/) · [ModelScope](https://modelscope.cn/models/inclusionAI/Ming-3.0-tts-0.5b)


In [ ]:
%pip install -q "transformers==4.46.3" "accelerate==1.3.0" "diffusers==0.33.0" "x_transformers==2.9.2" "soundfile==0.12.1" huggingface_hub onnxruntime inflect einops loguru tqdm packaging

# Ming-3.0-tts pins transformers==4.46.3 (the checkpoint was saved with that
# release). Do not pin torch / torchaudio: use Colab's CUDA stack.
# flash-attn and pynini are optional in the README; skip both on Colab.
# pynini has no reliable pip wheel; we pass text_normalization=False later.

import os
import subprocess

REPO = "/content/Ming-3.0-tts" if os.path.isdir("/content") else os.path.abspath("Ming-3.0-tts")
COMMIT = "028c1eace193e7acdb256453da89b6e9207e052b"  # 2026-10-21: 0.5B split + voice consistency

if not os.path.isdir(os.path.join(REPO, ".git")):
    subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/inclusionAI/Ming-3.0-tts.git", REPO])
subprocess.check_call(["git", "fetch", "--depth", "1", "origin", COMMIT], cwd=REPO)
subprocess.check_call(["git", "checkout", "--detach", COMMIT], cwd=REPO)
print("Ming-3.0-tts", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip())
print("REPO", REPO)


> **Restart the runtime now:** *Runtime > Restart session*, then continue from the next cell.
> Colab has already imported its preinstalled transformers. The 4.46.3 pin is invisible until the process restarts. The clone on disk survives the restart.


In [ ]:
import os
import subprocess

import torch
import transformers
from packaging.version import Version

if Version(transformers.__version__).major != 4:
    raise SystemExit(
        f"transformers {transformers.__version__} is loaded; Ming-3.0-tts pins 4.46.3. "
        "Run the install cell, then Runtime > Restart session."
    )
assert Version(transformers.__version__) >= Version("4.46.3"), (
    f"transformers {transformers.__version__} is loaded; Ming-3.0-tts needs 4.46.3+. "
    "Run the install cell, then Runtime > Restart session."
)

if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

# T4 (sm_75) has no FlashAttention 2. Keep PyTorch SDPA on the mem-efficient and math kernels.
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

major, minor = torch.cuda.get_device_capability()
NATIVE_BF16 = major >= 8
# T4 has no native bfloat16. float16 last-layer hidden states overflow (max 65504)
# and poison CFG even when DiT / VAE weights are already float32. Load as float16
# to keep Colab system RAM down, then run the whole generate path in float32.
LOAD_DTYPE = torch.bfloat16 if NATIVE_BF16 else torch.float16
RUN_DTYPE = torch.bfloat16 if NATIVE_BF16 else torch.float32
LLM_DTYPE = RUN_DTYPE
AUDIO_DTYPE = RUN_DTYPE
DTYPE = RUN_DTYPE
REPO = "/content/Ming-3.0-tts" if os.path.isdir("/content/Ming-3.0-tts") else os.path.abspath("Ming-3.0-tts")
print("transformers", transformers.__version__)
print("torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(), f"sm_{major}{minor}")
print("load dtype:", LOAD_DTYPE, "| generate / run dtype:", RUN_DTYPE)
print("autocast on generate path:", "on (bf16)" if NATIVE_BF16 else "OFF (T4 float32)")
print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print("REPO", REPO)


## 1. Float32 generate path on T4

`MingTTS.__init__` and `BailingMMNativeForConditionalGeneration.generate` hardcode `torch.bfloat16`. Restore the three files from git, then apply a **targeted** rewrite — not a blanket `bfloat16` → `float16` replace.

On a T4 the previous split (fp16 LLM + fp32 audio, autocast off) still produced a non-finite waveform after ~350 steps. Module dtypes were already correct; the leak was the **hidden-state / VAE dtype**, not the parameter dtype:

- Qwen last-layer hidden states in float16 overflow. `z_diff.to(float32)` after the fact cannot recover Inf, so CFG (`pred + (pred - null) * 2`) and the VAE decoder stay NaN.
- `generate()` wraps sample + VAE decode in autocast. That must stay **disabled**.
- The audio tokenizer `Linear` used to cast **weights to `x.dtype`**. A float16 latent then silently ran the decoder in float16 even though `mm.audio` reported float32.

So on compute capability &lt; 8.0 this cell now:

- Loads the checkpoint as **float16** (Colab system RAM), then upcasts the **entire** model — Qwen2.5-0.5B, DiT / flow, VAE, aggregator, speaker / stop heads — to **float32**.
- Turns **autocast off** around the LLM step, `flowloss.sample`, and `audio.decode`.
- Casts hidden states to float32 **before** the flow head, forces latents to float32 before VAE decode, and stops the VAE `Linear` from downcasting weights.
- Prints guardrails on the first decode step (`z_diff` dtype, finite flag, module dtypes, autocast).

On an L4 or A100 the sources stay on upstream bfloat16.

The first run downloads about 3.0 GB from `inclusionAI/Ming-3.0-tts-0.5B`. `text_normalization=False` skips `pynini`. The audio tokenizer checkpoint pins `flash_attention_2`; the GitHub config already downgrades that to SDPA when `flash-attn` is missing.


In [ ]:
import os
import random
import subprocess
import sys

import numpy as np
import torch

# Re-apply on every run so a leftover T4 patch does not stick on an A100.
PATCH_FILES = ("ming_tts.py", "modeling_bailingmm.py", "audio_tokenizer/audio_encoder.py")
subprocess.check_call(["git", "checkout", "--", *PATCH_FILES], cwd=REPO)


def rewrite(rel, pairs):
    path = os.path.join(REPO, rel)
    src = open(path).read()
    for needle, repl in pairs:
        if needle not in src:
            raise SystemExit(f"{rel} no longer contains:\n{needle}\n(pin may be stale)")
        src = src.replace(needle, repl)
    open(path, "w").write(src)
    print("patched", rel, f"({len(pairs)} edit(s))")


if NATIVE_BF16:
    print("native bfloat16: leaving ming_tts.py / modeling_bailingmm.py / audio_encoder.py unchanged")
else:
    # Load the 0.5B checkpoint as float16 (system RAM), then run generate in float32.
    rewrite(
        "ming_tts.py",
        [
            (
                "local_path, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True,",
                "local_path, torch_dtype=torch.float16, low_cpu_mem_usage=True,",
            ),
            (
                ").eval().to(torch.bfloat16).to(device)",
                ").eval().to(torch.float16).to(device)",
            ),
        ],
    )
    rewrite(
        "modeling_bailingmm.py",
        [
            ("self.llm_dytpe = torch.bfloat16", "self.llm_dytpe = torch.float32"),
            # generate() wraps sample() + VAE decode; sample() wraps the LLM step.
            # Nested float16 autocast is what overflowed CFG / decode on T4.
            (
                "with torch.autocast(device_type='cuda', dtype=torch.bfloat16):",
                "with torch.autocast(device_type='cuda', enabled=False):",
            ),
            (
                "waveform = prompt_waveform.to(device=self.device, dtype=torch.bfloat16)",
                "waveform = prompt_waveform.to(device=self.device, dtype=torch.float32)",
            ),
            (
                "            past_key_values = outputs.past_key_values\n"
                "            z_diff = outputs.hidden_states[-1][:, -1:, :]\n",
                "            past_key_values = outputs.past_key_values\n"
                "            # T4: fp16 last-layer states overflow (max 65504) and poison CFG.\n"
                "            z_diff = outputs.hidden_states[-1][:, -1:, :].float()\n"
                "            if step == 0:\n"
                "                print(\n"
                "                    \"generate guardrails: z_diff\", z_diff.dtype,\n"
                "                    \"finite\", bool(torch.isfinite(z_diff).all()),\n"
                "                    \"| flowloss\", next(self.flowloss.parameters()).dtype,\n"
                "                    \"| VAE\", next(self.audio.parameters()).dtype,\n"
                "                    \"| autocast\", torch.is_autocast_enabled(),\n"
                "                )\n"
                "            if not torch.isfinite(z_diff).all():\n"
                "                raise RuntimeError(f\"LLM hidden state is non-finite at step {step}\")\n",
            ),
            (
                "                    1, self.history_patch_size, self.latent_dim, device=z_diff.device, dtype=z_diff.dtype",
                "                    1, self.history_patch_size, self.latent_dim, device=z_diff.device, dtype=torch.float32",
            ),
            (
                "                    prompt_latent = prompt_latent.to(z_diff.dtype)",
                "                    prompt_latent = prompt_latent.float()",
            ),
            (
                "            sampled_token_latent = self.flowloss.sample(\n"
                "                z_diff, latent_history, cfg, self.patch_size, sigma=sigma, temperature=temperature\n"
                "            )",
                "            audio_dtype = torch.float32\n"
                "            with torch.autocast(device_type='cuda', enabled=False):\n"
                "                sampled_token_latent = self.flowloss.sample(\n"
                "                    z_diff.to(audio_dtype), latent_history.to(audio_dtype),\n"
                "                    cfg, self.patch_size, sigma=sigma, temperature=temperature\n"
                "                )\n"
                "            if not torch.isfinite(sampled_token_latent).all():\n"
                "                raise RuntimeError(f\"flowloss.sample produced NaN/Inf at step {step}\")",
            ),
            (
                "            if self.stop_head(z_diff)[0][0].softmax(dim=-1)[1] > 0.5 and step > 3:",
                "            if self.stop_head(z_diff.to(self.stop_head.weight.dtype))[0][0].softmax(dim=-1)[1] > 0.5 and step > 3:",
            ),
            (
                "            inputs_embeds = self.linear_proj_audio(sampled_token_latent)",
                "            inputs_embeds = self.linear_proj_audio(\n"
                "                sampled_token_latent.to(next(self.linear_proj_audio.parameters()).dtype)\n"
                "            )\n"
                "            inputs_embeds = inputs_embeds.to(self.model.get_input_embeddings().weight.dtype)",
            ),
            (
                "                se = self.spk_head(se.to(self.device))",
                "                se = self.spk_head(se.to(device=self.device, dtype=self.spk_head.weight.dtype))",
            ),
            (
                "            projected_latent = prompt_latent.reshape(-1, self.patch_size, self.latent_dim)",
                "            projected_latent = prompt_latent.float()\n"
                "            projected_latent = projected_latent.reshape(-1, self.patch_size, self.latent_dim)",
            ),
            (
                "            sampled_tokens = torch.cat(sampled_tokens_list, dim=1)\n"
                "            speech = self.audio.decode(\n"
                "                sampled_tokens, past_key_values=None, use_cache=False, last_chunk=True\n"
                "            )[0]",
                "            if not sampled_tokens_list:\n"
                "                raise RuntimeError(\"generate() sampled 0 tokens\")\n"
                "            sampled_tokens = torch.cat(sampled_tokens_list, dim=1).float()\n"
                "            if not torch.isfinite(sampled_tokens).all():\n"
                "                raise RuntimeError(\"generate() latents contain NaN/Inf before VAE decode\")\n"
                "            with torch.autocast(device_type='cuda', enabled=False):\n"
                "                speech = self.audio.decode(\n"
                "                    sampled_tokens, past_key_values=None, use_cache=False, last_chunk=True\n"
                "                )[0]",
            ),
        ],
    )
    rewrite(
        "audio_tokenizer/audio_encoder.py",
        [
            (
                "        return F.linear(\n"
                "            x,\n"
                "            self.weight.to(x.dtype),\n"
                "            None if self.bias is None else self.bias.to(x.dtype),\n"
                "        )",
                "        # Keep matmuls in the module dtype (fp32 on T4). Casting weights to\n"
                "        # x.dtype silently ran the VAE decoder in float16 when a latent leaked.\n"
                "        return F.linear(\n"
                "            x.to(self.weight.dtype),\n"
                "            self.weight,\n"
                "            None if self.bias is None else self.bias,\n"
                "        )",
            ),
        ],
    )

# Drop a cached import so re-running this cell picks up the rewritten sources.
for _mod in (
    "ming_tts",
    "modeling_bailingmm",
    "fm.dit",
    "fm.flowloss",
    "fm.CFM",
    "audio_tokenizer.audio_encoder",
    "audio_tokenizer.modeling_audio_vae",
):
    sys.modules.pop(_mod, None)

if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.chdir(REPO)

from ming_tts import MingTTS

MODEL_ID = "inclusionAI/Ming-3.0-tts-0.5B"

# infer.py default seed.
random.seed(1895)
np.random.seed(1895)
torch.manual_seed(1895)
torch.cuda.manual_seed_all(1895)

model = MingTTS(MODEL_ID, device="cuda:0", text_normalization=False)

mm = model.model
if not NATIVE_BF16:
    # Full generate path in float32. Mixed fp16-LLM / fp32-audio still NaN'd:
    # last-layer hidden states overflowed before they reached the flow head.
    mm.to(RUN_DTYPE)
    print("T4 generate path: upcasted entire model to", RUN_DTYPE)

print("sample_rate", model.sample_rate)
print("LLM", next(mm.model.parameters()).dtype)
for name in ("audio", "flowloss", "linear_proj_audio", "spk_head", "stop_head"):
    print(name, next(getattr(mm, name).parameters()).dtype)
print("generate dtype:", RUN_DTYPE, "| autocast enabled:", torch.is_autocast_enabled())
print("VAE Linear keeps weight dtype (no silent fp16 downcast)")
print(f"VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")


## 2. Built-in English voice

`task="IP"` selects a catalog voice. No reference clip. `use_zero_spk_emb=True` is the cookbook's recipe for a built-in voice. Recommended English names are `US_James`, `US_Stella`, `US_Emma`, `UK_George`, and `UK_Charlotte`. This cell uses **US_James** (male, American, narration).

The waveform is also saved and reused as the clone reference in the next section (generate-a-reference path from the model card).


In [ ]:
import time

import torchaudio
from IPython.display import Audio, display

RUNS = []
REF_TEXT = "Good evening. The garden will remain open until sunset, provided the weather stays clear."
REF_PATH = "/tmp/ming_ref_us_james.wav"


def as_wave(waveform):
    waveform = waveform.detach().cpu().float()
    if waveform.ndim == 1:
        waveform = waveform.unsqueeze(0)
    return waveform


def speak(title, **kwargs):
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    waveform = model.synthesize(**kwargs)
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    wave = as_wave(waveform)
    finite = bool(torch.isfinite(wave).all())
    print(
        f"{title}: finite={finite} min={wave.min().item():.4f} max={wave.max().item():.4f} "
        f"dtype={wave.dtype} | generate={RUN_DTYPE} autocast={torch.is_autocast_enabled()}"
    )
    if not finite:
        raise RuntimeError(
            f"{title}: waveform contains NaN/Inf. On a T4 the whole generate path "
            "(LLM hidden states, DiT / flow, VAE decode) must stay float32 with autocast off."
        )
    duration = wave.shape[-1] / model.sample_rate
    allocated = torch.cuda.memory_allocated() / 1024**3
    peak = torch.cuda.max_memory_allocated() / 1024**3
    print(
        f"{title}: {duration:.2f}s audio in {elapsed:.1f}s "
        f"({duration / elapsed:.2f}x) | VRAM {allocated:.2f} GB allocated, {peak:.2f} GB peak"
    )
    display(Audio(wave.numpy(), rate=model.sample_rate))
    RUNS.append((title, duration, elapsed, allocated, peak))
    return wave


ref = speak(
    "US_James (built-in)",
    task="IP",
    text=REF_TEXT,
    language_id="English",
    controls={"ip": "US_James"},
    use_zero_spk_emb=True,
)
torchaudio.save(REF_PATH, ref, sample_rate=model.sample_rate)
print("saved reference", REF_PATH)


## 3. Zero-shot voice cloning

`task="TTS"` plus a reference wav **and** its transcript. The cookbook is explicit: the model conditions on the pair, so a wrong transcript hurts similarity. `use_spk_emb=True` also injects the CAM++ speaker embedding from `campplus.onnx`.

The reference is the US_James line from the previous cell — generated by the model itself, so the transcript is exact. Do not point this at a recording of a real person without their consent.


In [ ]:
speak(
    "zero-shot clone",
    task="TTS",
    text="Cloning needs only a few seconds of audio and the matching transcript. This line should follow the voice you just heard.",
    language_id="English",
    prompt_wav_path=REF_PATH,
    prompt_text=REF_TEXT,
    use_spk_emb=True,
)


## 4. Voice Design

`task="VOICE_DESIGN"` builds a speaker from a natural-language description. No reference audio. `use_zero_spk_emb=True` again. `raw_text=True` sends the sentence as one piece — the cookbook notes that the style applies to the whole utterance, so it should not be split.

`controls["style"]` is the description. English and Chinese descriptions are both accepted.


In [ ]:
speak(
    "voice design",
    task="VOICE_DESIGN",
    text="Far below the ice, the glacier keeps a slow record of the changing climate.",
    language_id="English",
    controls={
        "style": (
            "A mature, steady male documentary narrator. Clear diction, "
            "unhurried pace, calm and thoughtful, close to the microphone."
        )
    },
    use_zero_spk_emb=True,
    raw_text=True,
)


## 5. Emotion, speed, and inline tags

These are three separate tasks. The cookbook rejects combining emotion or speed with Voice Design, a built-in voice, or a free-form instruction.

- `EMOTION` — `controls["emotion"]` is one of `高兴` (happy), `悲伤` (sad), `愤怒` (angry), `恐惧` (fearful), `厌恶` (disgusted), `惊讶` (surprised), `中性` (neutral).
- `BASIC` — `speed` is `慢速` / `中速` / `快速`; `pitch` and `volume` are `低` / `中` / `高`. Pass all three; they move independently.
- Inline tags sit in the spoken text on `task="TTS"`. Tags are case-sensitive and need the brackets: `[Laughter]`, `[Sigh]`, `[Breathing]`, `[Cough]`, `[Uhm]`, and the surprise / question set in the cookbook.

The emotion and speed calls reuse the generated US_James reference so the voice stays constant while the delivery changes.


In [ ]:
speak(
    "emotion happy",
    task="EMOTION",
    text="Every test passed on the first try. We can finally share the good news!",
    language_id="English",
    controls={"emotion": "高兴"},
    prompt_wav_path=REF_PATH,
    prompt_text=REF_TEXT,
    use_spk_emb=True,
)

speak(
    "speed fast",
    task="BASIC",
    text="Tonight's observation will start once the clouds break. Please wait for the next update.",
    language_id="English",
    controls={"speed": "快速", "pitch": "中", "volume": "中"},
    prompt_wav_path=REF_PATH,
    prompt_text=REF_TEXT,
    use_spk_emb=True,
)

speak(
    "inline tags",
    task="TTS",
    text="I checked the drawer twice [Sigh] and it was empty [Laughter] again.",
    language_id="English",
    prompt_wav_path=REF_PATH,
    prompt_text=REF_TEXT,
    use_spk_emb=True,
)


## 6. One non-English sample

`task="MULTILINGUAL"` is required for German, Spanish, French, Korean, and Japanese. `language_id` both writes `Target language: …` into the prompt and keeps the text away from the Chinese/English normalizer (which would strip Japanese kana). Short codes (`es`) and full names (`Spanish`) both work.

The voice is carried by the reference, not the language, so the English US_James clip can speak Spanish.


In [ ]:
speak(
    "Spanish",
    task="MULTILINGUAL",
    text="El jardín permanece abierto hasta el atardecer, si el tiempo se mantiene despejado.",
    language_id="Spanish",
    prompt_wav_path=REF_PATH,
    prompt_text=REF_TEXT,
    use_spk_emb=True,
)


## 7. VRAM and timing

Allocated memory after load is the full float32 generate path (Qwen2.5-0.5B + DiT / VAE) — about 6 GB on a T4, vs ~4.6 GB for the earlier fp16-LLM / fp32-audio split that still NaN'd. Peak is the decode step. RTF is audio seconds per wall-clock second; above 1.0 means faster than real time.


In [ ]:
header = f"{'clip':<28} {'audio_s':>8} {'wall_s':>8} {'rtf':>6} {'alloc_GB':>9} {'peak_GB':>8}"
print(header)
print("-" * 72)
for title, duration, elapsed, allocated, peak in RUNS:
    rtf = duration / elapsed if elapsed else 0.0
    print(f"{title:<28} {duration:8.2f} {elapsed:8.1f} {rtf:6.2f} {allocated:9.2f} {peak:8.2f}")
print()
print("GPU:", torch.cuda.get_device_name())
print("generate / run dtype:", RUN_DTYPE)
print("LLM dtype:", next(model.model.model.parameters()).dtype)
print("audio / DiT dtype:", next(model.model.audio.parameters()).dtype)
print("autocast enabled:", torch.is_autocast_enabled())
print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"VRAM allocated now: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
